# NHL Draft Analysis - Determining the Performance Score

The goal of this notebook is to define a performance metric that fairly captures NHL player value for use in draft evaluation.

## Setup

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nhl_drafts_clean.db")
print("Connected!")

Connected!


In [2]:
# Load full dataset
df = pd.read_sql_query("""
    SELECT draft_year, round, pick_number, drafted_by, player_name, 
           position, games_played, goals, assists, points, penalty_minutes
    FROM nhl_draft_picks
    ORDER BY draft_year, pick_number
""", conn)

df['games_played'] = df['games_played'].fillna(0)
df['points'] = df['points'].fillna(0)

print(f"Loaded {len(df)} players")
df.head()

Loaded 2766 players


,draft_year,round,pick_number,drafted_by,player_name,position,games_played,goals,assists,points,penalty_minutes
0,2005,1,1,Pittsburgh Penguins,Sidney Crosby,C,1417.0,653.0,1103.0,1756.0,896.0
1,2005,1,2,Anaheim Ducks,Bobby Ryan,R,866.0,261.0,308.0,569.0,470.0
2,2005,1,3,Carolina Hurricanes,Jack Johnson,D,1228.0,77.0,265.0,342.0,639.0
3,2005,1,4,Minnesota Wild,Benoit Pouliot,L,625.0,130.0,133.0,263.0,371.0
4,2005,1,5,Montreal Canadiens,Carey Price,G,712.0,0.0,13.0,13.0,51.0


## The Formula

The goal of this analysis is to measure team draft effectiveness — how well teams perform relative to the value of the picks they are given.

To do this, we need to compare each draft pick to a reasonable expectation. A 1st overall pick is expected to produce far more than a 6th round pick, so we evaluate teams based on how their selections perform relative to what is typical for that draft slot.

### Defining Player Performance

Before we can evaluate draft picks, we need a single metric that captures a player’s NHL career value.

The two statistics available are:

* Career games played
* Career points

Each tells part of the story:

* Games played captures longevity, reliability, and role consistency
* Points captures offensive production and high-end impact

Using only one would bias the evaluation:

Points alone would undervalue defensemen and depth players
Games played alone would fail to separate elite players from average ones

To balance this, we combine both into a single metric:

**Performance Score = games_played + (points × X)**

Where X determines how much weight points should carry relative to games played.

Rather than choosing X arbitrarily, we derive it directly from the data by making the average contribution of both components equal:

* avg_games_played = avg_points × X
* **X = avg_games_played / avg_points**

This ensures that, across the dataset, games played and points contribute equally to the overall performance score.

The performance score calculated here is the foundation for all downstream analysis. Expected performance by draft position and team-level draft value are calculated in `methodology.ipynb`

### Important Note

Goalies are excluded from this analysis.

Because goalies accumulate very few points regardless of their actual impact, they would systematically receive low performance scores. This would unfairly penalize teams for drafting goalies early, even when those were strong decisions.

Excluding them ensures a more consistent and fair comparison across teams, though it is acknowledged as a limitation.

In [3]:
nhl_players = df[(df['games_played'] > 0) & (df['points'] > 0)]
avg_gp = nhl_players['games_played'].mean()
avg_pts = nhl_players['points'].mean()
natural_X = round(avg_gp / avg_pts, 2)

print(f"Average GP:                {avg_gp:.1f}")
print(f"Average Points:            {avg_pts:.1f}")
print(f"Natural equal-weight X:    {natural_X}")

Average GP:                370.1
Average Points:            161.5
Natural equal-weight X:    2.29


With a natural equal-weight X of 2.29, both components contribute equally 
to the performance score on average. This is our starting point — derived 
directly from the data with no assumptions about whether longevity or 
scoring is more valuable.

Before committing to this value we run two checks:
1. **Positional Fairness** — does X=2.29 significantly underrepresent 
   defensemen relative to what the actual drafts would suggest?
2. **Sensitivity** — if we deviated one step in either direction, would 
   it meaningfully change the rankings?

If both checks pass, X=2.29 is confirmed as the final coefficient.

## Two Checks Before We Commit

Throughout both checks we evaluate the hindsight top 30 — representing 
roughly one pick per team per draft, the most scrutinized tier of any 
draft class and the range where teams invest the most scouting resources.

### Check 1 - Positional Fairness

We compare the positional breakdown of the hindsight top 30 at three values 
of X against the actual top 30 picks across all 13 drafts. The three values 
tested are X=1.29, X=2.29, and X=3.29 — one step below, the center, and 
one step above, where the gap is derived as 1.0.

If X=2.29 significantly underrepresents defensemen relative to the actual 
baseline — we define significant as more than 5 percentage points, or roughly 
2 extra forwards per draft class — we would consider moving to a lower value 
of X to improve positional fairness at the cost of equal weighting.

In [4]:
def show_position_breakdown(df, x_values, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']

    # Baseline — actual top N picks positional breakdown
    top_n_actual = df[df['pick_number'] <= top_n].copy()
    top_n_actual['position_group'] = top_n_actual['position'].apply(
        lambda x: 'D' if x == 'D' else 'F'
    )
    baseline = top_n_actual.groupby('position_group').size().reset_index(name='count')
    baseline['percentage'] = (baseline['count'] / len(top_n_actual) * 100).round(1)
    baseline = baseline.set_index('position_group')['percentage']

    print(f"Positional breakdown of hindsight top {top_n} vs actual top {top_n} picks\n")
    print(f"{'X Value':<12} {'F %':<10} {'D %':<10} {'F % actual':<15} {'D % actual':<15} {'F diff':<10} {'D diff'}")
    print("-" * 85)

    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df['hindsight_rank'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')

        top_n_df = df[df['hindsight_rank'] <= top_n].copy()
        top_n_df['position_group'] = top_n_df['position'].apply(
            lambda x: 'D' if x == 'D' else 'F'
        )

        breakdown = top_n_df.groupby('position_group').size().reset_index(name='count')
        total = breakdown['count'].sum()
        breakdown['percentage'] = (breakdown['count'] / total * 100).round(1)
        breakdown = breakdown.set_index('position_group')['percentage']

        f_pct = breakdown.get('F', 0)
        d_pct = breakdown.get('D', 0)
        f_actual = baseline.get('F', 0)
        d_actual = baseline.get('D', 0)
        f_diff = round(f_pct - f_actual, 1)
        d_diff = round(d_pct - d_actual, 1)

        print(f"X={X:<10} {f_pct:<10} {d_pct:<10} {f_actual:<15} {d_actual:<15} {f_diff:<10} {d_diff}")

`show_position_breakdown` takes the full dataset and a list of X values 
and compares the positional breakdown of the hindsight top 30 against the 
actual top 30 picks across all 13 drafts. It reports the forward and 
defenseman percentages at each X value alongside the actual baseline and 
the difference between them.

In [5]:
def show_rank_movement(df, year, x_values, top_n=30):
    df = df.copy()
    df = df[df['position'] != 'G']
    
    rank_data = {}
    for X in x_values:
        df['performance_score'] = df['games_played'] + (df['points'] * X)
        df[f'rank_X{X}'] = df.groupby('draft_year')['performance_score'].rank(ascending=False, method='min')
        rank_data[X] = df[df['draft_year'] == year][['player_name', 'position', 'pick_number', f'rank_X{X}']].copy()
    
    merged = rank_data[x_values[0]][['player_name', 'position', 'pick_number', f'rank_X{x_values[0]}']].copy()
    for X in x_values[1:]:
        merged = merged.merge(rank_data[X][['player_name', f'rank_X{X}']], on='player_name')
    
    rank_cols = [f'rank_X{X}' for X in x_values]
    merged = merged[merged[rank_cols].min(axis=1) <= top_n]
    
    # Calculate difference from center value (X=2.29)
    center_col = f'rank_X{x_values[1]}'
    for col in [f'rank_X{x_values[0]}', f'rank_X{x_values[2]}']:
        merged[f'diff_{col}'] = (merged[col] - merged[center_col]).abs()
    
    merged['avg_diff_from_center'] = merged[[f'diff_rank_X{x_values[0]}', 
                                              f'diff_rank_X{x_values[2]}']].mean(axis=1).round(1)
    merged['avg_rank'] = merged[rank_cols].mean(axis=1)
    merged = merged.sort_values('avg_rank')
    
    merged.columns = ['player', 'pos', 'actual_pick'] + \
                     [f'X={X}' for X in x_values] + \
                     [f'diff_from_center_low', 'diff_from_center_high', 
                      'avg_diff_from_center', 'avg_rank']
    merged = merged.reset_index(drop=True)
    
    avg_overall = merged['avg_diff_from_center'].mean().round(2)
    print(f"{year} Draft — Rank Movement (center = X=2.29)")
    print(merged[['player', 'pos', 'actual_pick', 
                  f'X={x_values[0]}', f'X={x_values[1]}', f'X={x_values[2]}',
                  'avg_diff_from_center']].to_string())
    print(f"\nAverage rank difference from X=2.29: {avg_overall} spots")

`show_rank_movement` takes the full dataset, a draft year, and a list of 
X values and shows how each player in the top 30 ranks at each coefficient, 
with X=2.29 as the anchor. The `avg_diff_from_center` column reports the 
average number of spots each player moves when shifting one step in either 
direction from the center value.

In [6]:
gap = 1.0
x_values = [round(natural_X - gap, 2), natural_X, round(natural_X + gap, 2)]
print(f"Test values: {x_values}")

Test values: [1.29, 2.29, 3.29]


In [7]:
show_position_breakdown(df, x_values)

Positional breakdown of hindsight top 30 vs actual top 30 picks

X Value      F %        D %        F % actual      D % actual      F diff     D diff
-------------------------------------------------------------------------------------
X=1.29       70.0       30.0       66.8            33.2            3.2        -3.2
X=2.29       70.8       29.2       66.8            33.2            4.0        -4.0
X=3.29       70.8       29.2       66.8            33.2            4.0        -4.0


The forward over-representation at X=2.29 is 4.0 percentage points — below 
our threshold of 5 percentage points. Across all 13 drafts this equates to 
roughly 16 additional forwards in the hindsight top 30, or just over one 
extra forward per draft class. 

**Check 1 passed.** X=2.29 does not introduce unacceptable positional bias.

### Check 2 — Sensitivity

We now look at how much the rankings change when we move one step in either 
direction from X=2.29. We use three benchmark drafts — 2005, 2007, and 2015 
— chosen because they contain well known players whose careers are easy to 
evaluate intuitively.

We measure sensitivity as the average rank difference from X=2.29 across 
all players in the top 30. If the average difference is small, the choice 
of X is not consequential and X=2.29 is confirmed.

In [8]:
show_rank_movement(df, 2005, x_values)
show_rank_movement(df, 2007, x_values)
show_rank_movement(df, 2015, x_values)

2005 Draft — Rank Movement (center = X=2.29)
                   player pos  actual_pick  X=1.29  X=2.29  X=3.29  avg_diff_from_center
0           Sidney Crosby   C            1     1.0     1.0     1.0                   0.0
1            Anze Kopitar   C           11     2.0     2.0     2.0                   0.0
2       Kristopher Letang   D           62     3.0     3.0     3.0                   0.0
3            Paul Stastny   C           44     4.0     4.0     4.0                   0.0
4              T.J. Oshie   R           24     6.0     5.0     5.0                   0.5
5            Keith Yandle   D          105     5.0     6.0     6.0                   0.5
6         Andrew Cogliano   C           25     7.0     7.0     7.0                   0.0
7     Marc-Edouard Vlasic   D           35     8.0     8.0    11.0                   1.5
8              Bobby Ryan   R            2    11.0     9.0     8.0                   1.5
9        Patric Hornqvist   R          230    10.0    11.0    10.

Across the three benchmark drafts, the average rank difference from X=2.29 
when moving one full gap unit in either direction was 0.75, 0.50, and 0.80 
spots respectively. Given that a gap of 1.0 represents a 44% deviation from 
the equal-weight center, these differences are remarkably small. The most 
sensitive player across all three drafts was Kirill Kaprizov in 2015 with 
an average difference of 3.5 spots — an outlier explained by his exceptional 
scoring rate relative to his games played.

**Check 2 passed.** Even substantial deviations from X=2.29 produce 
negligible changes in the rankings, confirming the methodology is robust.

## The Decision

Both checks passed. X=2.29 is confirmed as the final coefficient.

This value is derived directly from the data as the natural equal-weight 
point between games played and points. It makes no philosophical assumptions 
about whether longevity or scoring is more valuable. The positional fairness 
check confirmed it does not significantly underrepresent defensemen. The 
sensitivity check confirmed that deviating from it would not meaningfully 
change the conclusions.

The final performance score formula is:

**Performance Score = games_played + (points * 2.29)**

## Adjusting for Career Length

There is one more problem to address before we apply the formula.

The dataset spans 13 draft years — from 2005 to 2017. A player drafted in 2005 has had nearly two decades to accumulate games played and points. A player drafted in 2017 has had roughly eight seasons. Because the performance score is additive — every game played and every point scored increases it — players from earlier draft years are systematically rewarded simply for having more time, not necessarily for being better draft picks.

This is a fairness problem. We are comparing players across draft years, and the metric needs to put them on equal footing.

The ideal fix would be to evaluate every player on their first eight seasons only. Our dataset contains career totals rather than season-by-season splits, so we can't do that exactly — but we can approximate it by capping each player's games played contribution at a fixed ceiling before calculating their performance score.

The question is: what should that ceiling be?

We want the cap to reflect what a successful player from the most recent draft class has actually had time to accumulate — not the median or average, which would be pulled down by players who never stuck in the league. We're interested in the upper range of what 2017 draftees have genuinely played, so we use the **75th percentile of games played among 2017 draftees with at least one NHL game**.

Using the median or mean would cap too aggressively — we'd be throwing away real, meaningful career data from established players for no good reason. The 75th percentile gives us a ceiling that's grounded in what the most successful recent draftees have actually achieved, while still protecting them from being unfairly compared against veterans with a decade more runway.

In [9]:
# Temporarily exclude goalies to investigate the 2017 cohort
df_skaters = df[df['position'] != 'G'].copy()
df_2017 = df_skaters[df_skaters['draft_year'] == 2017]
played_2017 = df_2017[df_2017['games_played'] > 0]['games_played']

print("Games played distribution — 2017 draftees with GP > 0:")
print(played_2017.describe().round(1))

cap = int(played_2017.quantile(0.75))
print(f"\n75th percentile: {cap} games")
print(f"This is our career length cap.")

Games played distribution — 2017 draftees with GP > 0:
count     92.0
mean     212.7
std      178.7
min        1.0
25%       39.5
50%      191.5
75%      356.8
max      603.0
Name: games_played, dtype: float64

75th percentile: 356 games
This is our career length cap.


With the cap established, we prorate each player's performance score to the capped window. For players whose career falls within the cap, nothing changes — their score is untouched. For players who have exceeded the cap, their performance score is scaled down proportionally to what they would have produced in that window.

> Adjusted Performance Score = performance_score × min(1, cap / games_played)

A player with 1400 games and a performance score of 5406 gets scaled to the equivalent of their first ~350 games. A player with 300 games is untouched. This keeps the metric fair across draft years without discarding real data.

One important note: players with zero games played remain at zero — the proration only applies to players who actually appeared in the NHL.

## Applying the Formula

With X=2.29 confirmed, we now apply the formula to the full dataset. 
For each draft year we calculate the performance score for every player.

In [10]:
FINAL_X = 2.29

df_scores = df[df['position'] != 'G'].copy()

# Step 1: Apply performance score formula
df_scores['performance_score'] = (
    df_scores['games_played'] + (df_scores['points'] * FINAL_X)
).round(2)

# Step 2: Prorate to career length cap
df_scores['performance_score'] = df_scores.apply(
    lambda row: round(row['performance_score'] * min(1, cap / row['games_played']), 2)
    if row['games_played'] > 0 else 0.0,
    axis=1
)

print(f"Formula applied and prorated to {cap}-game cap for {len(df_scores)} players")
print(f"\nTop 10 by performance score after proration:")
print(df_scores.nlargest(10, 'performance_score')[
    ['draft_year', 'pick_number', 'player_name', 'games_played', 'performance_score']
].to_string(index=False))

Formula applied and prorated to 356-game cap for 2484 players

Top 10 by performance score after proration:
 draft_year  pick_number      player_name  games_played  performance_score
       2015            1   Connor McDavid         789.0            1604.17
       2011           58  Nikita Kucherov         873.0            1400.96
       2005            1    Sidney Crosby        1417.0            1366.28
       2014            3   Leon Draisaitl         855.0            1360.03
       2013            1 Nathan MacKinnon         944.0            1337.92
       2015          135  Kirill Kaprizov         393.0            1330.97
       2016            1  Auston Matthews         689.0            1278.91
       2014           25   David Pastrnak         828.0            1270.68
       2015            4     Mitch Marner         733.0            1265.78
       2015           10   Mikko Rantanen         711.0            1248.06


## Franchise Consolidation

The Atlanta Thrashers relocated to become the Winnipeg Jets in 2011, 
and the Phoenix Coyotes were rebranded as the Arizona Coyotes in 2014. 
Since the front offices moved with the franchises, their draft histories 
are combined under the current franchise name.

In [11]:
# Franchise consolidation
print("Unique teams before consolidation:")
print(sorted(df_scores['drafted_by'].unique()))
print(f"\nTotal teams: {df_scores['drafted_by'].nunique()}")

franchise_map = {
    'Atlanta Thrashers': 'Winnipeg Jets',
    'Phoenix Coyotes': 'Arizona Coyotes'
}

df_scores['drafted_by'] = df_scores['drafted_by'].replace(franchise_map)

print("\nUnique teams after consolidation:")
print(sorted(df_scores['drafted_by'].unique()))
print(f"\nTotal teams: {df_scores['drafted_by'].nunique()}")

Unique teams before consolidation:
['Anaheim Ducks', 'Arizona Coyotes', 'Atlanta Thrashers', 'Boston Bruins', 'Buffalo Sabres', 'Calgary Flames', 'Carolina Hurricanes', 'Chicago Blackhawks', 'Colorado Avalanche', 'Columbus Blue Jackets', 'Dallas Stars', 'Detroit Red Wings', 'Edmonton Oilers', 'Florida Panthers', 'Los Angeles Kings', 'Minnesota Wild', 'Montreal Canadiens', 'Nashville Predators', 'New Jersey Devils', 'New York Islanders', 'New York Rangers', 'Ottawa Senators', 'Philadelphia Flyers', 'Phoenix Coyotes', 'Pittsburgh Penguins', 'San Jose Sharks', 'St. Louis Blues', 'Tampa Bay Lightning', 'Toronto Maple Leafs', 'Vancouver Canucks', 'Vegas Golden Knights', 'Washington Capitals', 'Winnipeg Jets']

Total teams: 33

Unique teams after consolidation:
['Anaheim Ducks', 'Arizona Coyotes', 'Boston Bruins', 'Buffalo Sabres', 'Calgary Flames', 'Carolina Hurricanes', 'Chicago Blackhawks', 'Colorado Avalanche', 'Columbus Blue Jackets', 'Dallas Stars', 'Detroit Red Wings', 'Edmonton Oiler

In [12]:
# Output to csv
output_cols = ['draft_year', 'round', 'pick_number', 'drafted_by', 'player_name',
               'position', 'games_played', 'points', 'performance_score']

df_scores[output_cols].to_csv('../data/outputs/draft_scores.csv', index=False)
print(f"draft_scores.csv saved with {len(df_scores)} rows")

draft_scores.csv saved with 2484 rows


## End of Analysis

In [13]:
conn.close()
print("Connection closed.")

Connection closed.


The formula has been applied to all 13 drafts from 2005 to 2017. 
The output has been saved to `data/outputs/draft_scores.csv` and will serve as the input for `methodology.ipynb`, where expected performance and draft value are calculated for each player.